# 0. Import library

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("cuda is available")
else:
    print("cuda is NOT available")

import numpy as np
from tqdm import tqdm
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import time
import copy
from moving_average import moving_average_1d

import importlib
import policy
importlib.reload(policy)
from policy import PolicyNN

from nn_functions import surrogate

import sys
sys.path.append('../1_model')
from TiDE import TideModule, quantile_loss  


cuda is available


# 1-1. Import Data 

In [2]:
df_all = pd.read_csv('../0_data/merged_df_2_99_temp_depth.csv')
print(df_all.shape)
print(df_all.columns)

nan_rows = df_all[df_all.isna().any(axis=1)]

df_all = df_all.dropna()

loc_X = df_all["X"].to_numpy().reshape(-1,1)
loc_Y = df_all["Y"].to_numpy().reshape(-1,1)
loc_Z = df_all["Z"].to_numpy().reshape(-1,1)
dist_X = df_all["Dist_to_nearest_X"].to_numpy().reshape(-1,1)
dist_Y = df_all["Dist_to_nearest_Y"].to_numpy().reshape(-1,1)
# dist_Z = df_all["Dist_to_nearest_Z"].to_numpy()[::2].reshape(-1,1)
scan_spd = df_all["scanning_speed"].to_numpy().reshape(-1,1)
laser_power = df_all["Laser_power"].to_numpy().reshape(-1,1)
laser_on_off = df_all["laser_power_number"].to_numpy().reshape(-1,1)

# apply moving average for mp temp
mp_temp_raw = df_all["melt_pool_temperature"].to_numpy()
mp_temp_mv = moving_average_1d(mp_temp_raw,4)
mp_temp = copy.deepcopy(mp_temp_raw)
mp_temp[1:-2] = mp_temp_mv
mp_temp = mp_temp.reshape(-1,1)

# apply moving average for mp depth
mp_depth_raw = df_all["melt_pool_depth"].to_numpy()
mp_depth_mv = moving_average_1d(mp_depth_raw,4)
mp_depth = copy.deepcopy(mp_depth_raw)
mp_depth[1:-2] = mp_depth_mv
mp_depth = mp_depth.reshape(-1,1)       


(610615, 12)
Index(['time_index', 'melt_pool_temperature', 'melt_pool_depth',
       'scanning_speed', 'X', 'Y', 'Z', 'Dist_to_nearest_X',
       'Dist_to_nearest_Y', 'Dist_to_nearest_Z', 'Laser_power',
       'laser_power_number'],
      dtype='object')


## 1-2. Normalize data

In [3]:
# stack input array
x_original_scale = np.concatenate((loc_Z, dist_X, dist_Y, laser_power), axis=1)
y_original_scale = np.concatenate((mp_temp, mp_depth), axis=1)

# scaling
x_max = np.max(x_original_scale,0).reshape(1,-1)
x_min = np.min(x_original_scale,0).reshape(1,-1) 
y_max = np.max(y_original_scale,0).reshape(1,-1) 
y_min = np.min(y_original_scale,0).reshape(1,-1)

print("x_max:", np.round(x_max, 3).tolist())
print("x_min:", np.round(x_min, 3).tolist())
print("y_max:", np.round(y_max, 3).tolist())
print("y_min:", np.round(y_min, 3).tolist())

x_max: [[7.5, 20.0, 20.0, 732.298]]
x_min: [[0.0, 0.75, 0.75, 504.26]]
y_max: [[4509.855, 0.551]]
y_min: [[436.608, -0.559]]


In [4]:
class scalers():
    def __init__(self,x_max, x_min, y_max, y_min) -> None:
        self.x_max = x_max
        self.x_min = x_min
        self.y_max = y_max
        self.y_min = y_min
        
        return None
    
    def scaler_x(self, x_original, dim_id = -1):
        if dim_id == -1:
            x_s = -1 + 2 * ((x_original - self.x_min) / (self.x_max-self.x_min))
            return x_s
        else: 
            x_s = -1 + 2 * (x_original - self.x_min[0,dim_id]) / (self.x_max[0,dim_id] - self.x_min[0,dim_id])
            return x_s
    
    def inv_scaler_x(self, x_s, dim_id = -1):
        
        if dim_id == -1:
            x_original = (x_s + 1)*0.5*(self.x_max-self.x_min) + self.x_min
            return x_original
        else: 
            x_original = (x_s + 1)*0.5*(self.x_max[0,dim_id] - self.x_min[0,dim_id]) + self.x_min[0,dim_id]
            return x_original
        
    def scaler_y(self, y_original):
        return -1 + 2 * ((y_original - self.y_min) / (self.y_max-self.y_min))
    
    def inv_scaler_y(self, y_s):
        return (y_s + 1)*0.5*(self.y_max-self.y_min) + self.y_min

In [5]:
scaler = scalers(x_max, x_min, y_max, y_min)

x_s = scaler.scaler_x(x_original_scale)
y_s = scaler.scaler_y(y_original_scale)

print("x_s range:", np.min(x_s), "to", np.max(x_s))
print("y_s range:", np.min(y_s), "to", np.max(y_s))

print("x_s shape:", x_s.shape)
print("y_s shape:", y_s.shape)

x_s range: -1.0 to 1.0
y_s range: -1.0 to 1.0
x_s shape: (610417, 4)
y_s shape: (610417, 2)


## 1-3. Generate data

In [6]:
length = y_s.shape[0]

y_s_ref = np.random.uniform(0.0, 1.0, size=(length, 1))

e = 0.0001
y_depth_low = np.random.uniform(0.1423-e, 0.1423+e, size = (length,1))
y_depth_up = np.random.uniform(0.4126-e, 0.4126+e, size = (length,1))
# constraints!
#y_depth_low = 0*np.random.uniform(0.0, 0.5, size=(length, 1))
#y_depth_up = 0*np.random.uniform(0.5, 1.0, size=(length, 1))

y_s_const = np.concatenate((y_depth_low, y_depth_up), axis=1)

print("y_s_ref shape:", y_s_ref.shape)
print("y_s_const shape:", y_s_const.shape)

y_s_ref shape: (610417, 1)
y_s_const shape: (610417, 2)


## 1-4. Split data

In [7]:
# 기본 분할
cutoff_index = int(np.round(0.8 * x_s.shape[0]))
x_train, y_train = x_s[:cutoff_index], y_s[:cutoff_index]
x_val, y_val = x_s[cutoff_index:], y_s[cutoff_index:]
# y_ref_train, y_ref_val = y_s_ref[:cutoff_index], y_s_ref[cutoff_index:]
y_const_train, y_const_val = y_s_const[:cutoff_index], y_s_const[cutoff_index:]

window = 50
P = 50

# ------------------ Training set ------------------
n_train = cutoff_index - P - window
x_past_train = np.empty((n_train, window, 4))
y_past_train = np.empty((n_train, window, 2))
x_future_train = np.empty((n_train, P, 3))
y_ref_train_seq = np.empty((n_train, P, 1))
y_const_train_seq = np.empty((n_train, P, 2))

for i in tqdm(range(window, cutoff_index - P)):
    j = i - window
    x_past_train[j] = x_train[i-window:i]

    # shift for u past last component
    #u_last_shift = torch.empty(1).uniform_(-0.1, 0.1).item()
    #x_past_train[j, -1, 3] += u_last_shift

    y_past_train[j] = y_train[i-window:i]
    x_future_train[j] = x_train[i:i+P, :3]         

    # shift for temp future entire horizon
    # scalar_shift = torch.empty(1).uniform_(-0.1, 0.1).item()  # Random scalar shift for y_ref
    # y_ref_train_seq[j] = y_train[i:i+P, :1] + scalar_shift
    # y_ref_train_seq[j] = y_ref_train[i:i+P]

    # 조건에 따라 다른 범위의 shift 적용(0801)
    if y_train[i, 0].item() < 0.3:
        scalar_shift = torch.empty(1).uniform_(-0.0001, 0.0001).item()
    else:
        scalar_shift = torch.empty(1).uniform_(-0.0001, 0.0001).item()

    y_ref_train_seq[j] = y_train[i:i+P, :1] + scalar_shift

    y_const_train_seq[j] = y_const_train[i:i+P]

# ------------------ Validation set ------------------
val_cutoff = x_s.shape[0] - cutoff_index
n_val = val_cutoff - P - window
x_past_val = np.empty((n_val, window, 4))
y_past_val = np.empty((n_val, window, 2))
x_future_val = np.empty((n_val, P, 3))
y_ref_val_seq = np.empty((n_val, P, 1))
y_const_val_seq = np.empty((n_val, P, 2))

for i in tqdm(range(window, val_cutoff - P)):
    j = i - window
    x_past_val[j] = x_val[i-window:i]
    y_past_val[j] = y_val[i-window:i]
    x_future_val[j] = x_val[i:i+P, :3]

    # scalar_shift = torch.empty(1).uniform_(-0.1, 0.1).item() 
    # y_ref_val_seq[j] = y_val[i:i+P, :1] + scalar_shift
    # y_ref_val_seq[j] = y_ref_val[i:i+P]

    # 조건에 따라 다른 범위의 shift 적용 (0801)
    if y_val[i, 0].item() < 0.3:
        scalar_shift = torch.empty(1).uniform_(-0.0001, 0.0001).item()
    else:
        scalar_shift = torch.empty(1).uniform_(-0.0001, 0.0001).item()
    y_ref_val_seq[j] = y_val[i:i+P, :1] + scalar_shift


    y_const_val_seq[j] = y_const_val[i:i+P]

x_past_train = torch.tensor(x_past_train, dtype=torch.float32)
y_past_train = torch.tensor(y_past_train, dtype=torch.float32)
x_future_train = torch.tensor(x_future_train, dtype=torch.float32)
y_ref_train_seq = torch.tensor(y_ref_train_seq, dtype=torch.float32)
y_const_train_seq = torch.tensor(y_const_train_seq, dtype=torch.float32)

print("x_past shape : ", x_past_train.shape)       # (n_train, 50, 4)
print("y_past shape : ",y_past_train.shape)       # (n_train, 50, 2)
print("x_future shape : ",x_future_train.shape)     # (n_train, 50, 3)
print("y_ref shape : ",y_ref_train_seq.shape)    # (n_train, 50, 1)
print("y_const shape : ",y_const_train_seq.shape)  # (n_train, 50, 2)

x_past_val = torch.tensor(x_past_val, dtype=torch.float32)
y_past_val = torch.tensor(y_past_val, dtype=torch.float32)
x_future_val = torch.tensor(x_future_val, dtype=torch.float32)
y_ref_val_seq = torch.tensor(y_ref_val_seq, dtype=torch.float32)
y_const_val_seq = torch.tensor(y_const_val_seq, dtype=torch.float32)

print("x_past_val shape : ", x_past_val.shape)       # (n_val, 50, 4)
print("y_past_val shape : ", y_past_val.shape)       # (n_val, 50, 2)
print("x_future_val shape : ", x_future_val.shape)   # (n_val, 50, 3)
print("y_ref_val shape : ", y_ref_val_seq.shape)     # (n_val, 50, 1)
print("y_const_val shape : ", y_const_val_seq.shape) # (n_val, 50, 2)

100%|██████████| 121983/121983 [00:01<00:00, 98125.55it/s] 


x_past shape :  torch.Size([488234, 50, 4])
y_past shape :  torch.Size([488234, 50, 2])
x_future shape :  torch.Size([488234, 50, 3])
y_ref shape :  torch.Size([488234, 50, 1])
y_const shape :  torch.Size([488234, 50, 2])
x_past_val shape :  torch.Size([121983, 50, 4])
y_past_val shape :  torch.Size([121983, 50, 2])
x_future_val shape :  torch.Size([121983, 50, 3])
y_ref_val shape :  torch.Size([121983, 50, 1])
y_const_val shape :  torch.Size([121983, 50, 2])


In [8]:
import torch
from torch.utils.data import TensorDataset
from sequence_vae import SequenceVAE, combine_inputs, train_vae

# === 1. 데이터 결합 ===
X_train = combine_inputs(
    x_past_train, y_past_train, x_future_train, y_ref_train_seq, y_const_train_seq
)
X_val = combine_inputs(
    x_past_val, y_past_val, x_future_val, y_ref_val_seq, y_const_val_seq
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

train_dataset = TensorDataset(X_train)
val_dataset = TensorDataset(X_val)

X_train: torch.Size([488234, 50, 12])
X_val: torch.Size([121983, 50, 12])


# 4. Train

In [9]:

# === 2. 모델 설정 ===
input_dim = X_train.shape[-1]  # 12
model = SequenceVAE(
    input_dim=input_dim,
    hidden_dim=128,
    latent_dim=16,
    num_layers=2
)

# === 3. 학습 ===
trained_model = train_vae(
    model,
    train_dataset,
    val_dataset,
    num_epochs=30,
    lr=1e-3,
    batch_size=256,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

# === 4. latent 추출 ===
device = 'cuda' if torch.cuda.is_available() else 'cpu'
trained_model = trained_model.to(device)
trained_model.eval()

z_list = []
batch_size = 256  

with torch.no_grad():
    for i in range(0, X_val.shape[0], batch_size):
        batch = X_val[i:i+batch_size].to(device)
        mu, logvar = trained_model.encoder(batch)
        z = trained_model.reparameterize(mu, logvar)
        z_list.append(z.cpu())  
        torch.cuda.empty_cache()  

z_all = torch.cat(z_list, dim=0)
print("Latent representation shape:", z_all.shape)

# === 5. 모델 저장 ===
import os
os.makedirs("checkpoints", exist_ok=True)
torch.save(trained_model.state_dict(), "checkpoints/vae_128H_16Z_epoch30_case0.pth")
print("✅ 모델 저장 완료: checkpoints/vae_128H_16Z_epoch20.pth")


[Epoch 1] Train Loss: 0.0478 | Val Loss: 0.0093
[Epoch 2] Train Loss: 0.0053 | Val Loss: 0.0046
[Epoch 3] Train Loss: 0.0038 | Val Loss: 0.0031
[Epoch 4] Train Loss: 0.0031 | Val Loss: 0.0030
[Epoch 5] Train Loss: 0.0031 | Val Loss: 0.0026
[Epoch 6] Train Loss: 0.0023 | Val Loss: 0.0024
[Epoch 7] Train Loss: 0.0031 | Val Loss: 0.0022
[Epoch 8] Train Loss: 0.0022 | Val Loss: 0.0023
[Epoch 9] Train Loss: 0.0022 | Val Loss: 0.0020
[Epoch 10] Train Loss: 0.0021 | Val Loss: 0.0025
[Epoch 11] Train Loss: 0.0021 | Val Loss: 0.0388
[Epoch 12] Train Loss: 0.0034 | Val Loss: 0.0021
[Epoch 13] Train Loss: 0.0020 | Val Loss: 0.0019
[Epoch 14] Train Loss: 0.0020 | Val Loss: 0.0019
[Epoch 15] Train Loss: 0.0048 | Val Loss: 0.0026
[Epoch 16] Train Loss: 0.0039 | Val Loss: 0.0032
[Epoch 17] Train Loss: 0.0024 | Val Loss: 0.0024
[Epoch 18] Train Loss: 0.0023 | Val Loss: 0.0023
[Epoch 19] Train Loss: 0.0020 | Val Loss: 0.0027
[Epoch 20] Train Loss: 0.0020 | Val Loss: 0.0021
[Epoch 21] Train Loss: 0.0019